# **AI Engineer 2 - LLM Task**


**Task Pertama:**
- Jalankan model LLM open-source secara lokal menggunakan HuggingFace Transformers
(contoh: Mistral atau LLaMA-2).
- Buat script sederhana yang menerima prompt dari user dan mengembalikan respons
dari model.
- Uji coba dua use-case ringan: tanya jawab (QA) dan ringkasan teks pendek.
- Penerapan Teknik Prompting dengan membandingkan jenis prompt: zero-shot dan
few-shot dan lihat pengaruh dari kualitas jawaban.
- Bungkus pipeline dengan API sederhana menggunakan FastAPI atau Flask.
- Jalankan API tersebut secara lokal
- Tambahkan logging sederhana: tampilkan waktu respons dan jumlah permintaan di
terminal/log file.
- Buat antarmuka sederhana menggunakan Streamlit atau Gradio untuk input/output
prompt.

In [11]:
# Import Libraries

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import time

In [2]:
# Load model dan tokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Loading model... (this may take a while)")
base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token 

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model.to(device)

print("Model loaded successfully!")


Loading model... (this may take a while)
Model loaded successfully!


In [12]:
# ========== MODEL RUNNER ==========
def run_model(prompt, model, tokenizer, max_new_tokens=500, temperature=0.7, top_p=0.9):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
        return_attention_mask=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    elapsed = time.time() - start_time
    
    return response, elapsed

# ========== QA FUNCTION ==========
def run_qa(prompt, model, tokenizer, max_new_tokens=500):

    return run_model(prompt, model, tokenizer, max_new_tokens=max_new_tokens)

# ========== SUMMARY FUNCTIONS ==========
def run_summary(prompt, model, tokenizer, max_new_tokens=500):
    return run_model(prompt, model, tokenizer, max_new_tokens=max_new_tokens)


In [4]:
# === Use Case: QA - Zero Shot on Campus Location ===

question = "Di mana lokasi kampus ITB Jatinangor?"
qa_zero_shot_prompt = f"""Question: {question}
Answer:"""

qa_zero, t1 = run_qa(qa_zero_shot_prompt, base_model, base_tokenizer)

print("\n=== QA Zero-Shot Test (Base Model) ===")
print(f"Question: {question}")
print(f"Answer: {qa_zero.replace(qa_zero_shot_prompt, '').strip()} (Time: {t1:.2f}s)")


=== QA Zero-Shot Test (Base Model) ===
Question: Di mana lokasi kampus ITB Jatinangor?
Answer: Kampus ITB Jatinangor adalah sebuah jenjang studi yang dibentuk pada tingkat universitas. Semua bidang yang ditetapkan oleh program studi dan semakin banyaknya, maka jenis-jenis perpustakaan itu juga berbeda dengan lainnya. (Time: 21.28s)


In [5]:
# === Use Case: QA - Few Shot on Campus Location ===

question = "Di mana lokasi kampus ITB Jatinangor?"
qa_few_shot_prompt = f"""
Jawablah pertanyaan-pertanyaan berikut tentang lokasi universitas.

Q: Di mana kampus utama Universitas Harvard?
A: Kampus utama Universitas Harvard terletak di Cambridge, Massachusetts.

Q: Di kota mana Universitas Oxford berada?
A: Universitas Oxford berada di kota Oxford, Inggris.

Q: Di mana lokasi Universitas Stanford?
A: Universitas Stanford terletak di Stanford, California, dekat Palo Alto.

Q: {question}
A:"""

qa_few, t2 = run_qa(qa_few_shot_prompt, base_model, base_tokenizer)

print("\n=== QA Few-Shot Test (Base Model) ===")
print(f"Question: {question}")
print(f"Answer: {qa_few.replace(qa_few_shot_prompt, '').strip()} (Time: {t2:.2f}s)")



=== QA Few-Shot Test (Base Model) ===
Question: Di mana lokasi kampus ITB Jatinangor?
Answer: Lokasi kampus Ibuku ITB Jatinangor adalah jl. Raya Abepura No.105/937, Kabupaten Tuban, Daerah Khusus Ibukota Jakarta Timur. (Time: 16.81s)


In [ ]:
fine_tuned_model_path = "../models/fine-tuned-QA-Bot"
print(f"\n--- 3. Testing the Fine-Tuned Expert Model from {fine_tuned_model_path} ---")

# Load model dan tokenizer yang telah di-fine-tune
ft_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path)
ft_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
ft_model.to(device)

def get_answer_from_ft_model(question, max_tokens=500):
    prompt = f"Question: {question} Answer:"
    
    response_raw, elapsed_time = run_qa(
        prompt=prompt,
        model=ft_model,
        tokenizer=ft_tokenizer,
        max_new_tokens=max_tokens
    )
    
    
    cleaned_response = response_raw.replace(prompt, "").strip()
    
    return cleaned_response, elapsed_time


# Example Test Cases - Known Questions (from training data)
print("\n--- Testing Known Questions ---")
test_questions_known = [
    "Where is the Jatinangor campus of ITB located?",
    "What city is home to Universitas Gadjah Mada (UGM)?",
    "In which city can I find Universitas Airlangga (Unair)?"
]

for q in test_questions_known:
    print(f"Q: {q}")
    answer, time_taken = get_answer_from_ft_model(q)
    print(f"A: {answer} (Time: {time_taken:.2f}s)\n")


# Example Test Cases - Unknown Questions (to check generalization)
print("\n--- Testing Unknown Questions (to check generalization) ---")
test_questions_unknown = [
    "Where is the IPB University business school located?",
    "Which faculties are located at the UI Salemba campus?",
    "How many main campuses does Unair have in Surabaya?"
]

for q in test_questions_unknown:
    print(f"Q: {q}")
    answer, time_taken = get_answer_from_ft_model(q)
    print(f"A: {answer} (Time: {time_taken:.2f}s)\n")


# Example Test Cases - Wrong/External Question
print("\n--- Testing Wrong/External Question ---")
test_question_wrong = [
    "Where is the National University of Singapore located?"
]

for q in test_question_wrong:
    print(f"Q: {q}")
    answer, time_taken = get_answer_from_ft_model(q)
    print(f"A: {answer} (Time: {time_taken:.2f}s)\n")


--- 3. Testing the Fine-Tuned Expert Model from ./fine-tuned-Campus-Bot ---


The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.



--- Testing Known Questions ---
Q: Where is the Jatinangor campus of ITB located?
A: The Jatinangor campus of ITB is located in Sumedang Regency, West Java. (Time: 6.71s)

Q: What city is home to Universitas Gadjah Mada (UGM)?
A: Universitas Gadjah Mada is located in Yogyakarta. (Time: 3.62s)

Q: In which city can I find Universitas Airlangga (Unair)?
A: Universitas Airlangga (Unair) is located in Surabaya, East Java, with campuses spread across the city. (Time: 6.39s)


--- Testing Unknown Questions (to check generalization) ---
Q: Where is the IPB University business school located?
A: The business school of IPB University is located in Pangrung, Depok. (Time: 4.25s)

Q: Which faculties are located at the UI Salemba campus?
A: The faculties that are located at the UI Salemba campus are Law, Humanities (Faculty of Social Sciences), and Science (Faculty of Natural Sciences). (Time: 8.36s)

Q: How many main campuses does Unair have in Surabaya?
A: Unair has one main campus in Surabaya.

In [13]:
# === Use Case 2: Summarization - Zero Shot==

text_to_summarize = """Universitas Brawijaya (UB) adalah universitas negeri yang terletak di kota Malang, Jawa Timur. Dikenal dengan lingkungan kampusnya yang hijau dan luas, UB menawarkan berbagai program studi di banyak fakultas, mulai dari kedokteran dan teknik hingga ilmu sosial dan pertanian, menjadikannya pilihan populer bagi mahasiswa dari seluruh Indonesia."""

summary_zero_shot_prompt = f"""Ringkas paragraf berikut:

{text_to_summarize}

Ringkasan:"""


summary_zero, t3 = run_summary(summary_zero_shot_prompt, base_model, base_tokenizer)

print("\n=== Summarization Zero-Shot Test (Base Model) ===")
print(f"Text: {text_to_summarize}")
print(f"Summary: {summary_zero.replace(summary_zero_shot_prompt, '').strip()} (Time: {t3:.2f}s)")


=== Summarization Zero-Shot Test (Base Model) ===
Text: Universitas Brawijaya (UB) adalah universitas negeri yang terletak di kota Malang, Jawa Timur. Dikenal dengan lingkungan kampusnya yang hijau dan luas, UB menawarkan berbagai program studi di banyak fakultas, mulai dari kedokteran dan teknik hingga ilmu sosial dan pertanian, menjadikannya pilihan populer bagi mahasiswa dari seluruh Indonesia.
Summary: Universitas Brawijaya merupakan univeritas negeri yang terletak di kabupaten Malang, Jawa Timur. Kami mengucapkan kepada para mahasiswi untuk membuat permintaan aplikasi web di website ini. Dan ketika semua permintaan sudah ditandai dengan status "sudah lulus", maka kami akan memproses permintaan dan memberikan hasil transaksi alamat email Anda sebelum tanggal 25 Oktober 2019. Untuk melakukan penyimpanan jurnal buku-buku di database lainnya, silahkan hubungi pihak admin kami di jl. Pantai Cendrawasih No.76A Malang, Jawa Timur 65143 at +62 812 55022323. (Time: 54.96s)


In [14]:
# === Use Case 2: Summarization - Few Shot ===

text_to_summarize = """Universitas Brawijaya (UB) adalah universitas negeri yang terletak di kota Malang, Jawa Timur. Dikenal dengan lingkungan kampusnya yang hijau dan luas, UB menawarkan berbagai program studi di banyak fakultas, mulai dari kedokteran dan teknik hingga ilmu sosial dan pertanian, menjadikannya pilihan populer bagi mahasiswa dari seluruh Indonesia."""

summary_few_shot_prompt = f"""
Ringkas paragraf berikut dalam satu kalimat.

Teks: Universitas Harvard adalah universitas riset swasta Ivy League di Cambridge, Massachusetts. Didirikan pada tahun 1636 dan dinamai dari dermawan pertamanya, pendeta Puritan John Harvard, universitas ini merupakan institusi pendidikan tinggi tertua di Amerika Serikat.
Ringkasan: Universitas Harvard adalah universitas tertua di Amerika Serikat yang terletak di Cambridge, Massachusetts.

Teks: Universitas Oxford adalah universitas riset kolese di Oxford, Inggris. Bukti pengajaran tercatat sejak tahun 1096, menjadikannya universitas tertua di dunia berbahasa Inggris dan tertua kedua di dunia yang masih beroperasi.
Ringkasan: Universitas Oxford di Inggris adalah universitas tertua di dunia berbahasa Inggris.

Teks: Universitas Stanford, yang terletak di Stanford, California, dikenal karena kekuatan akademisnya, kekayaannya, kedekatannya dengan Silicon Valley, dan peringkatnya sebagai salah satu universitas top dunia. Ini adalah universitas riset swasta.
Ringkasan: Universitas Stanford adalah universitas riset swasta terkemuka di California yang dikenal karena hubungannya dengan Silicon Valley.

Teks: {text_to_summarize}
Ringkasan:"""


summary_few, t4 = run_summary(summary_few_shot_prompt, base_model, base_tokenizer)

print("\n=== Summarization Few-Shot Test (Base Model) ===")
print(f"Text: {text_to_summarize}")
print(f"Summary: {summary_few.replace(summary_few_shot_prompt, '').strip()} (Time: {t4:.2f}s)")


=== Summarization Few-Shot Test (Base Model) ===
Text: Universitas Brawijaya (UB) adalah universitas negeri yang terletak di kota Malang, Jawa Timur. Dikenal dengan lingkungan kampusnya yang hijau dan luas, UB menawarkan berbagai program studi di banyak fakultas, mulai dari kedokteran dan teknik hingga ilmu sosial dan pertanian, menjadikannya pilihan populer bagi mahasiswa dari seluruh Indonesia.
Summary: Universitas Brawijaya menjadi universitas negeri yang paling populernya di Indonesia. (Time: 16.07s)


In [12]:
fine_tuned_model_path = "../models/fine-tuned-Summarization-Bot"
print(f"\n--- 3. Testing the Fine-Tuned Summarization Model from {fine_tuned_model_path} ---")

ft_model = AutoModelForCausalLM.from_pretrained(fine_tuned_model_path)
ft_tokenizer = AutoTokenizer.from_pretrained(fine_tuned_model_path)
ft_model.to(device)


def get_summary(text_to_summarize, max_new_tokens=500):
    prompt = f"Summarize the following text:\n\n{text_to_summarize}\n\nSummary:"
    
    response_raw, elapsed_time = run_summary(
        prompt=prompt,
        model=ft_model,
        tokenizer=ft_tokenizer,
        max_new_tokens=max_new_tokens
    )
    
    try:
        summary = response_raw.split("Summary:")[1].strip()
    except IndexError:
        summary = "Model failed to generate a valid summary."
        
    return summary, elapsed_time

# Example Test Cases - Known Text
print("\n--- Testing Known Text (from training data) ---")
test_texts_known = [
    "Institut Teknologi Bandung (ITB) features its main and most historic campus on Jalan Ganesha, Bandung, West Java. To accommodate growth, it has expanded with a secondary campus in Jatinangor, Sumedang Regency, which houses several of a newer faculties and programs.",
    "Universitas Airlangga (Unair) is a prominent university located in Surabaya, East Java. Its facilities are spread across three main campus locations within the city: Campus A for Medicine, Campus B for Social Sciences, and Campus C for Science and Technology."
]

for i, text in enumerate(test_texts_known):
    print(f"Text: {text}")
    summary, time_taken = get_summary(text)
    print(f"\nSummary:\n{summary} (Time: {time_taken:.2f}s)\n")


# Example Test Cases - Unknown Text
print("\n--- Testing Unknown Text (to check generalization) ---")
test_text_unknown = "Universitas Brawijaya (UB) is a major public university located in the city of Malang, East Java. Known for its green and spacious campus environment, it offers a wide range of programs across many faculties, from medicine and engineering to social sciences and agriculture, making it a popular choice for students across Indonesia."

print(f"\n[Unknown Text - Universitas Brawijaya]")
print(f"Text:{test_text_unknown}")
summary, time_taken = get_summary(test_text_unknown)
print(f"\nSummary:\n{summary} (Time: {time_taken:.2f}s)\n")


# Example Test Cases - External Text
print("\n--- Testing Wrong/External Text (to check robustness) ---")
test_text_external = "The Komodo dragon is a species of lizard found in the Indonesian islands of Komodo, Rinca, Flores, and Gili Motang. A member of the monitor lizard family, it is the largest living species of lizard, growing to a maximum length of 3 metres (10 ft) in rare cases and weighing up to approximately 70 kilograms (150 lb)."

print(f"Text:{test_text_external}")
summary, time_taken = get_summary(test_text_external)
print(f"\nSummary:\n{summary} (Time: {time_taken:.2f}s)\n")


--- 3. Testing the Fine-Tuned Summarization Model from ../models/fine-tuned-Summarization-Bot ---


The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers and GPU quantization are unavailable.



--- Testing Known Text (from training data) ---
Text: Institut Teknologi Bandung (ITB) features its main and most historic campus on Jalan Ganesha, Bandung, West Java. To accommodate growth, it has expanded with a secondary campus in Jatinangor, Sumedang Regency, which houses several of a newer faculties and programs.

Summary:
Institut Teknologi Bandung (ITB) has its primarycampus in Bandung and a newer, secondary campus for variousfaculties in Jatinangor, Sumedang. (Time: 9.12s)

Text: Universitas Airlangga (Unair) is a prominent university located in Surabaya, East Java. Its facilities are spread across three main campus locations within the city: Campus A for Medicine, Campus B for Social Sciences, and Campus C for Science and Technology.

Summary:
Universita Airlangga (Unair) is located in Surabaya and operates across three main campuses (A, B, and C), each focusing on different fields of study. (Time: 8.93s)


--- Testing Unknown Text (to check generalization) ---

[Unknown Text